# einops-rearrange composite — cx6: transpose with rearrange, then repeat along a new axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-rearrange`, `einops-repeat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-rearrange"
DD_ATOM_IDS = ["einops-rearrange", "einops-repeat"]
DD_SUBTOPICS = ["Einops: Rearrange", "Einops: Repeat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`einops-rearrange` does pure axis reordering / regrouping — the data doesn't change, only how axes are named. `'h w -> w h'` is a transpose. `einops-repeat` then introduces a new size that wasn't in the input, supplied as a kwarg.

**The composition.** Transpose `(H, W) -> (W, H)`, then tile across a new leading channel axis `(C, W, H)` — useful when you have a single greyscale image laid out `(H, W)` and need to feed a (C, H', W') CNN that expects channel-first AND the spatial order swapped (e.g. an x/y flip).

### Composite Exercise — transpose with rearrange, then repeat along a new axis

**Atoms exercised together**: `einops-rearrange`, `einops-repeat`

Build `cx6_transpose_then_channelize(x, channels)`.

Input: `x` of shape `(H, W)`. Output: a tensor of shape `(channels, W, H)` where:
- The spatial axes are swapped (transposed `H` and `W`).
- The new leading axis tiles `channels` copies of the transposed image.

Constraints:
- Step 1: ONE `rearrange` call to perform the `(H, W) -> (W, H)` transpose.
- Step 2: ONE `repeat` call to add the leading `channels` axis, producing `(channels, W, H)`.
- Every channel slice along axis 0 must equal the transposed image.
- No `.t()`, no `.transpose()`, no `.expand()`, no `torch.stack`.

In [ ]:
def cx6_transpose_then_channelize(x, channels):
    swapped = rearrange(x, 'h w -> w h')                  # rearrange atom
    return repeat(swapped, 'w h -> c w h', c=channels)    # repeat atom


<details><summary>Show solution — cx6</summary>

```python
def cx6_transpose_then_channelize(x, channels):
    swapped = rearrange(x, 'h w -> w h')                  # rearrange atom
    return repeat(swapped, 'w h -> c w h', c=channels)    # repeat atom
```

Two atoms, two calls. The transpose is the canonical `rearrange` use (reordering output axes); the leading-channel tile is the canonical `repeat` use (new axis introduced by kwarg). Swap the order (repeat-then-rearrange) and the rearrange pattern needs three axes instead of two — both work but the assigned shape contract is transpose-first.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx6',
        'subtopics': ["Einops: Rearrange", "Einops: Repeat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()